# 04 — Information limits (claims C3, C-M, I1)
Bayesian Cramér–Rao bounds of the production features at Sc1–Sc5 under three nuisance sets (inlet
known; inlet offsets + Ci + Q; plus every sensor calibration offset); which instrument limits β; what a
temperature-only monitor can know (masking); the sensor ladder S0–S4. numpy only, ~30 min.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
pts = {k: (scn.SCENARIOS[k].alpha, scn.SCENARIOS[k].beta) for k in scn.GATED + ["Sc5_severe_fouling"]}
NS = {"inlet_only": ("oTi", "oTci", "uCi", "uQ"), "all_offsets": tuple(info.NUISANCES)}
rows = []
for k, (a, b) in pts.items():
    for lab, nu in NS.items():
        r = info.bayes_crb(a, b, nuisances=nu, n=1500)
        rows.append({"scenario": k, "nuisances": lab, **{kk: v for kk, v in r.items() if isinstance(v, float)}})
crb = pd.DataFrame(rows); display(crb.round(4)); crb.to_csv(R / "04_crb.csv", index=False)

## Which instrument limits β? (each nuisance made known in turn, at Sc1)

In [ ]:
base = info.bayes_crb(1, 1, n=1500)
brk = {"all unknown": (base["sd_alpha"], base["sd_beta"])}
for k in info.NUISANCES:
    r = info.bayes_crb(1, 1, nuisances=tuple(x for x in info.NUISANCES if x != k), n=1500)
    brk[f"{k} known"] = (r["sd_alpha"], r["sd_beta"])
brk = pd.DataFrame(brk, index=["sd_alpha", "sd_beta"]).T; display(brk.round(4)); brk.to_csv(R / "04_instrument_breakdown.csv")

## Masking (C-M): information in a temperature-only monitor

In [ ]:
mask = {fs: info.bayes_crb(1, 1, feature_set=fs, n=1500, nuisances=("oTi", "oTci", "uCi", "uQ")) for fs in ["T_only", "C_T_only", "S1m"]}
display(pd.DataFrame(mask).T[["sd_alpha", "sd_beta"]].round(4)); da.save_json(R / "04_masking.json", mask)

## Sensor ladder (information side of notebook 10)

In [ ]:
lad = {s: info.bayes_crb(1, 1, sensors=s, n=1500) for s in sen.SENSORS}
display(pd.DataFrame(lad).T[["sd_alpha", "sd_beta", "corr_ab"]].round(4)); da.save_json(R / "04_sensor_ladder_crb.json", lad)
print(f"{(time.time() - T0) / 60:.1f} min")

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).